# Hacking Forums — 00: Reconocimiento del dataset

Este es el primer notebook del caso. Aquí no corremos ningún modelo de inteligencia artificial todavía:  
lo que hacemos es **entender qué tenemos, de dónde viene, y qué preguntas podemos responder** con esos datos.

Antes de escribir una sola línea de análisis, un investigador forense siempre hace reconocimiento:
¿qué archivos tengo? ¿cuántos registros hay? ¿en qué idioma están escritos? ¿qué calidad tienen los datos?

Ese es el trabajo de este notebook.

## ¿Qué son los hacking forums underground?

Los **foros underground** son comunidades en línea donde se intercambia información, herramientas y servicios  
relacionados con actividades ilegales en el ciberespacio: venta de credenciales robadas, herramientas de intrusión,  
accesos a sistemas comprometidos, y documentación de vulnerabilidades.

A diferencia de la dark web clásica (Tor), muchos de estos foros operaron o todavía operan en la internet  
superficial, protegidos por mecanismos de registro privado y moderación interna.

### ¿Por qué analizarlos juntos?

El dato más valioso no está dentro de un solo foro, sino **en las conexiones entre foros**:  
un actor que aparece en OGUsers, Cracked.to y Exploit.in bajo el mismo handle, o con el mismo estilo de escritura,  
es un candidato de atribución mucho más sólido que uno visto en un solo lugar.

El análisis comparativo cross-foro nos permite:
- Rastrear **persistencia de identidad** a lo largo del tiempo y de plataformas.
- Detectar **especialización por foro** (¿qué se vende en cada uno?).
- Identificar **actores pivote** que conectan ecosistemas diferentes.

## Los 7 foros evaluados inicialmente

Al inicio del caso evaluamos 7 dumps SQL de foros underground. No todos resultaron útiles.  
Usamos un sistema de **tiers** para clasificarlos según su valor analítico y su complejidad técnica:

### Tiers de utilidad

| Tier | Descripción |
|------|-------------|
| **A** | Dump SQL completo: usuarios + posts + threads + IPs. Permite análisis de red, atribución y estilometría. |
| **A↓** | Dump SQL parcial: solo usuarios. Sirve para pivoting de handles, pero sin contenido analizable. |
| **—** | Descartado: schema-only o muestra mínima sin valor analítico. |

### Tiers técnicos (formato del dump)

| Tier | Descripción |
|------|-------------|
| **B** | MyBB con prefijo no estándar. El parser detecta automáticamente el prefijo de tablas. |
| **C** | IPS 3.x — plataforma diferente a MyBB, con variante de prefijo (`ibf_` o sin prefijo). Requiere ingeniería inversa del schema. |

### Resultado de la evaluación

| Foro | Archivo | Parser | Utilidad | Técnica | Decisión |
|------|---------|--------|----------|---------|----------|
| **OGUsers** | OGUsers_2019.zip | MyBB | A | B | ✓ Incluido |
| **Exploit.in** | Exploit.in_2013.12.13.zip | IPS 3.x (ibf_) | A | C | ✓ Incluido (ruso) |
| **Cracked.to** | Cracked.to_2019.01.zip | MyBB | A | B | ✓ Incluido |
| **Nulled.io** | Nulled.io_2016.05.zip | IPS 3.x (sin prefijo) | A | C | ✓ Incluido (ver nota abajo) |
| **RaidForums** | RaidForums_2021.zip | MyBB | A | B | ✓ Incluido |
| **Hackforums.net** | Hackforums.net_2015.zip | MyBB | A↓ | B | ✗ Descartado (solo usuarios, sin posts) |
| **Void.to** | Void.to.zip | MyBB | — | B | ✗ Descartado (schema-only, cero datos) |

## El bug de Nulled.io: IPS 3.x sin prefijo → falso positivo en is_mybb()

**Este es el problema técnico más importante de la etapa de reconocimiento. Entendelo bien.**

Los foros pueden correr sobre distintas plataformas de software:
- **MyBB**: las tablas se llaman `mybb_users`, `mybb_posts`, `mybb_threads` (o con otro prefijo configurado).
- **IPS 3.x** (Invision Power Suite): las tablas se llaman `ibf_members`, `ibf_posts`, `ibf_topics`.

Nuestro parser usa una función `is_mybb()` que mira si el dump tiene tablas con prefijo tipo `mybb_`.  
El problema con **Nulled.io** es que su dump usa IPS 3.x, pero **sin el prefijo `ibf_`**:  
las tablas se llaman directamente `members`, `posts`, `topics`.

Cuando el parser ve un dump sin prefijo `ibf_`, asume que es MyBB → **falso positivo**.

### El fix

Al detectar Nulled.io, forzamos el modo IPS-sin-prefijo explícitamente en lugar de dejar que  
`is_mybb()` tome la decisión automáticamente. Esto se documenta en el código de carga del notebook 01.

```python
# Ejemplo conceptual del fix:
# En lugar de: dfs = load_forum(path)  # is_mybb() se confunde
# Hacemos:     dfs = load_forum(path, force_engine='ips_noprefix')
```

> **Por qué importa**: si cargás Nulled.io con el parser equivocado, las columnas no se mapean correctamente  
> y el análisis produce basura silenciosa — resultados que parecen válidos pero no lo son.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.utils import load_forum, list_forums, RESULTS_DIR, DATA_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

HF_DIR = DATA_DIR / 'Hacking Forums'
HF_RESULTS = RESULTS_DIR / 'hacking_forums'
HF_RESULTS.mkdir(parents=True, exist_ok=True)

FORUMS_WITH_POSTS = [
    "OGUsers_2019.zip",
    "Exploit.in_2013.12.13.zip",
    "Cracked.to_2019.01.zip",
    "Nulled.io_2016.05.zip",
    "RaidForums_2021.zip",
]

print(f"DATA_DIR:    {DATA_DIR}")
print(f"HF_DIR:      {HF_DIR}")
print(f"HF_RESULTS:  {HF_RESULTS}")

## Exploración básica de cada foro

Cargamos cada foro y preguntamos lo más básico:
- ¿Cuántas tablas tiene el dump?
- ¿Cuántos usuarios hay?
- ¿Cuántos posts hay?
- ¿Cuántos threads hay?

`load_forum()` es la función del proyecto que detecta automáticamente el formato del dump  
(MyBB, IPS 3.x con o sin prefijo) y devuelve un diccionario de DataFrames.

Un **DataFrame** es simplemente una tabla con filas y columnas, igual que una hoja de Excel.

In [ ]:
forums_data = {}

for fname in FORUMS_WITH_POSTS:
    path = HF_DIR / fname
    if not path.exists():
        print(f"  [NO ENCONTRADO] {fname}")
        continue
    try:
        dfs = load_forum(path)
        if not dfs:
            print(f"  [VACÍO]  {fname}: schema-only, sin datos")
            continue
        forums_data[Path(fname).stem] = dfs
        n_users   = len(dfs.get('user',   pd.DataFrame()))
        n_posts   = len(dfs.get('post',   pd.DataFrame()))
        n_threads = len(dfs.get('thread', pd.DataFrame()))
        n_tables  = len(dfs)
        print(f"  ✓  {Path(fname).stem:<30} tablas={n_tables}  usuarios={n_users:,}  posts={n_posts:,}  threads={n_threads:,}")
    except Exception as e:
        print(f"  ✗  {fname}: {e}")

print(f"\nForos cargados correctamente: {len(forums_data)}")

## Tabla comparativa de tamaño

Construimos una tabla resumen que compara todos los foros en las mismas dimensiones.

También extraemos el **rango de fechas** de los posts — esto nos dice cuándo estuvo activo cada foro  
y si los dumps son capturas de un momento puntual o de toda la historia del foro.

In [ ]:
rows = []
for name, dfs in forums_data.items():
    user_df   = dfs.get('user',   pd.DataFrame())
    post_df   = dfs.get('post',   pd.DataFrame())
    thread_df = dfs.get('thread', pd.DataFrame())

    date_start = date_end = '—'
    for col in ('dateline', 'post_date', 'date'):
        if col in post_df.columns:
            ts = pd.to_datetime(post_df[col], errors='coerce', utc=True)
            valid = ts.dropna()
            if len(valid):
                date_start = valid.min().strftime('%Y-%m')
                date_end   = valid.max().strftime('%Y-%m')
            break

    rows.append({
        'Foro':     name,
        'Usuarios': len(user_df),
        'Posts':    len(post_df),
        'Threads':  len(thread_df),
        'Primer post': date_start,
        'Último post': date_end,
    })

summary_df = pd.DataFrame(rows).set_index('Foro')
print(summary_df.to_string())

## Visualización: tamaño comparado

Los números en la tabla son informativos, pero es difícil comparar mentalmente  
"1.2 millones vs 800 mil". Un gráfico de barras hace esa comparación instantánea.

Generamos dos barras: posts por foro y usuarios por foro.

In [ ]:
import matplotlib.ticker as mticker

forum_names  = list(forums_data.keys())
short_names  = [n.split('_')[0] for n in forum_names]
post_counts  = [len(forums_data[f].get('post',   pd.DataFrame())) for f in forum_names]
user_counts  = [len(forums_data[f].get('user',   pd.DataFrame())) for f in forum_names]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars0 = axes[0].bar(short_names, post_counts, color='#4E9EE9', alpha=0.85)
axes[0].bar_label(bars0, fmt='{:,.0f}', padding=3, fontsize=8)
axes[0].set_title('Posts por foro')
axes[0].set_ylabel('Posts')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, _: f'{v/1_000_000:.1f}M' if v >= 1e6 else f'{v/1000:.0f}K'
))
axes[0].tick_params(axis='x', rotation=30)

bars1 = axes[1].bar(short_names, user_counts, color='#E9A24E', alpha=0.85)
axes[1].bar_label(bars1, fmt='{:,.0f}', padding=3, fontsize=8)
axes[1].set_title('Usuarios por foro')
axes[1].set_ylabel('Usuarios')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, _: f'{v/1000:.0f}K'
))
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Comparativa de tamaño entre foros', y=1.01)
plt.tight_layout()
plt.savefig(HF_RESULTS / 'reconocimiento_size_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Validación: ¿cuánto cargamos vs. cuánto debería haber?

Un dump SQL bien formado suele incluir al principio un comentario con el recuento original de filas  
de cada tabla. Podemos comparar ese número contra lo que cargamos.

Si hay una diferencia grande (digamos, >10%), puede indicar:
- Truncado del dump (el archivo fue cortado o comprimido con pérdida)
- Duplicados que el parser eliminó
- Posts eliminados antes del leak

La validación no es bloquente — si los números son razonablemente cercanos, continuamos.  
Si hay una diferencia masiva, lo documentamos como limitación del análisis.

> **Nota**: si el dump no incluye recuentos en sus comentarios, esta celda imprime los totales  
> cargados sin referencia de comparación — eso también es útil para documentar.

In [ ]:
# Recuentos esperados (documentados manualmente desde los comentarios de cada dump)
# Si un dump no tiene comentarios con recuentos, el valor es None
EXPECTED = {
    'OGUsers_2019':          {'users': None, 'posts': None},
    'Exploit.in_2013.12.13': {'users': None, 'posts': None},
    'Cracked.to_2019.01':    {'users': None, 'posts': None},
    'Nulled.io_2016.05':     {'users': None, 'posts': None},
    'RaidForums_2021':       {'users': None, 'posts': None},
}

print(f"{'Foro':<35} {'Usuarios cargados':>18} {'Posts cargados':>15} {'Nota'}")
print('-' * 80)
for name, dfs in forums_data.items():
    n_users = len(dfs.get('user', pd.DataFrame()))
    n_posts = len(dfs.get('post', pd.DataFrame()))
    exp = EXPECTED.get(name, {})
    exp_u = exp.get('users')
    exp_p = exp.get('posts')
    note = ''
    if exp_u is not None:
        diff_u = abs(n_users - exp_u) / exp_u * 100
        note += f'Δusuarios={diff_u:.1f}%  '
    if exp_p is not None:
        diff_p = abs(n_posts - exp_p) / exp_p * 100
        note += f'Δposts={diff_p:.1f}%'
    if not note:
        note = '(sin referencia en el dump)'
    print(f"{name:<35} {n_users:>18,} {n_posts:>15,} {note}")

## Detección de idioma por foro

**Este es el paso más importante antes de cualquier análisis de texto.**

Herramientas como BERTopic, NER y estilometría son **idioma-específicas**.  
Si aplicamos un pipeline entrenado en inglés sobre texto en ruso, obtenemos resultados sin sentido  
— y lo peor es que parecen válidos a simple vista.

La función `langdetect.detect()` analiza el texto y devuelve el código de idioma ISO 639-1:  
- `'en'` para inglés  
- `'ru'` para ruso  
- `'es'` para español, etc.

Internamente usa estadísticas de **n-gramas de caracteres** — no entiende el significado,  
solo reconoce los patrones de combinaciones de letras que son típicos de cada idioma.  
Por eso funciona bien con textos largos (más de 50 caracteres) y mal con textos muy cortos.

Muestreamos hasta **500 posts por foro** para mantener el análisis rápido.

In [ ]:
import re
import random
from langdetect import detect, LangDetectException

SAMPLE_PER_FORUM = 500

def strip_html(text):
    """Elimina etiquetas HTML del texto antes de detectar el idioma."""
    return re.sub(r"<[^>]+>", " ", str(text or ""))

lang_results = []

for name, dfs in forums_data.items():
    post_df  = dfs.get('post', pd.DataFrame())
    text_col = next((c for c in ('pagetext', 'message', 'post_content') if c in post_df.columns), None)
    if post_df.empty or text_col is None:
        print(f"[SKIP] {name}: sin posts o sin columna de texto")
        continue

    texts = post_df[text_col].dropna().astype(str).tolist()
    random.seed(42)
    sample = random.sample(texts, min(SAMPLE_PER_FORUM, len(texts)))

    lang_counts = {}
    for raw in sample:
        text = strip_html(raw)[:2000].strip()
        if len(text) < 30:
            continue
        try:
            lang = detect(text)
        except LangDetectException:
            lang = 'unknown'
        lang_counts[lang] = lang_counts.get(lang, 0) + 1

    total = sum(lang_counts.values()) or 1
    top_lang = max(lang_counts, key=lang_counts.get)
    top_pct  = lang_counts[top_lang] / total * 100

    for lang, count in sorted(lang_counts.items(), key=lambda x: -x[1])[:5]:
        lang_results.append({
            'foro': name.split('_')[0],
            'lang': lang,
            'pct': round(count / total * 100, 1),
        })

    flag = '⚠️  RUSO — pipeline separado' if top_lang == 'ru' else '✓  inglés'
    print(f"  {name.split('_')[0]:<15} → {top_lang} ({top_pct:.0f}%)  {flag}")

## Visualización de la distribución de idioma

El siguiente gráfico muestra, para cada foro, qué porcentaje de posts fueron detectados  
en cada idioma. Un foro con 80%+ en un idioma tiene un perfil lingüístico claro;  
uno más fragmentado puede tener secciones bilingües.

In [ ]:
if lang_results:
    lang_df = pd.DataFrame(lang_results)
    pivot = lang_df.pivot_table(index='foro', columns='lang', values='pct', fill_value=0)

    pivot.plot(kind='bar', figsize=(12, 5), colormap='tab10')
    plt.title('Distribución de idioma por foro (muestra de 500 posts)')
    plt.ylabel('% de posts detectados en ese idioma')
    plt.xlabel('')
    plt.xticks(rotation=30, ha='right')
    plt.legend(title='idioma', bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'reconocimiento_language_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Sin datos de idioma para visualizar.")

## Decisión de pipeline según idioma

Con la información de idioma, tomamos la decisión de qué herramientas usar en cada foro.

| Foro | Idioma dominante | Pipeline asignado |
|------|------------------|-------------------|
| OGUsers | Inglés (~95%+) | BERTopic + NER (qwen2.5:14b) + embeddings |
| Cracked.to | Inglés (~90%+) | BERTopic + NER + embeddings |
| Nulled.io | Inglés (~85%+) | BERTopic + NER + embeddings |
| RaidForums | Inglés (~90%+) | BERTopic + NER + embeddings |
| Exploit.in | Ruso (~80%+) | **Solo embeddings** (qwen3 multilingual); NER excluido |

> **Por qué excluimos NER en Exploit.in**: `qwen2.5:14b` fue entrenado principalmente en inglés.  
> Aplicarlo sobre texto ruso produce extracción de entidades con alta tasa de error.  
> El notebook 03 documenta el modelo ruso adecuado si se quisiera agregar esa capa.

## Preguntas de investigación

Cerramos este notebook con las tres preguntas que guían todo el análisis.  
Cada notebook siguiente responde parcialmente alguna de ellas.

### Pregunta 1 — ¿Hay usuarios que aparecen en múltiples foros?

Un usuario que registra el mismo handle en OGUsers, Cracked.to y Exploit.in  
es un candidato fuerte de atribución de identidad cross-foro.  
Respuesta en: **notebook 02** (pivoting exacto y fuzzy) + **notebook 03** (similitud de embeddings).

### Pregunta 2 — ¿Cada foro tiene una especialización temática detectable?

¿OGUsers habla principalmente de cuentas robadas? ¿RaidForums de accesos?  
¿Exploit.in de exploits técnicos? Topic modeling y TF-IDF lo revelan.  
Respuesta en: **notebook 02** (BERTopic) + **notebook 03** (TF-IDF comparado).

### Pregunta 3 — ¿Los actores que aparecen en múltiples foros tienen patrones de comportamiento cross-foro?

¿Un usuario que aparece en dos foros actúa como vendedor en uno y comprador en el otro?  
¿Su estilo de escritura es consistente? ¿Sus topics también?  
Respuesta en: **notebook 03** (perfilado de actores) + **notebook 04** (síntesis).

---

**Siguiente paso**: notebook `01_ingenieria_datos.ipynb` — carga, limpieza y preparación del corpus.